## Importanto os dados de treino e teste

In [1]:
import joblib

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
X_tr, X_te, y_tr, y_te = joblib.load('/content/drive/MyDrive/Colab Notebooks/dados_preparados.pkl')

## Importando os transformers

In [4]:
configs = joblib.load('/content/drive/MyDrive/Colab Notebooks/transformers.pkl')
print(type(configs))

<class 'dict'>


## Imports

In [5]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate

# Treinamento dos modelos

## SVM

Treinamento do modelo com as diferentes column transformers e combinações:
- txt_word - TF-IDF para palavras, com unigramas e bigramas
- txt_word + txt_char - TF-IDF para caracteres, com trigramas a quiquigramas
- txt_word + meta - TF-IDF para palavras e metadados estilométricos
- txt_word + txt_char + meta - Todos os passos combinados

In [6]:
import matplotlib.pyplot as plt
from sklearn.svm import LinearSVC
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import cross_val_predict, cross_validate

scoring = ["f1_macro", "accuracy", "precision_macro", "recall_macro"]
resultados = []
'''
print("SVM")
for nome, transformers in configs.items():
    pipe = Pipeline(
        [
            ("prep", ColumnTransformer(transformers)),
             ("svm", LinearSVC(class_weight="balanced", random_state=42, max_iter=2000))
        ]
    )

    # 1. Executa o cross_validate padrão para as métricas do seu relatório
    cv = cross_validate(pipe, X_tr, y_tr, cv=5, scoring=scoring)
    print(f"\n{nome}")

    linha = {"config": nome}
    for m in scoring:
        vals = cv[f"test_{m}"]
        print(f"  {m}: {vals.mean():.4f} ± {vals.std():.4f}")
        linha[f"{m}_mean"] = vals.mean()
        linha[f"{m}_std"] = vals.std()

    resultados.append(linha)

    # Matriz de confusão
    y_pred_cv = cross_val_predict(pipe, X_tr, y_tr, cv=5)

    cm = confusion_matrix(y_tr, y_pred_cv)

    fig, ax = plt.subplots(figsize=(5, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap="Blues", ax=ax)
    ax.set_title(f"Matriz de Confusão: {nome}")
    plt.show()
'''

'\nprint("SVM")\nfor nome, transformers in configs.items():\n    pipe = Pipeline(\n        [\n            ("prep", ColumnTransformer(transformers)),\n             ("svm", LinearSVC(class_weight="balanced", random_state=42, max_iter=2000))\n        ]\n    )\n\n    # 1. Executa o cross_validate padrão para as métricas do seu relatório\n    cv = cross_validate(pipe, X_tr, y_tr, cv=5, scoring=scoring)\n    print(f"\n{nome}")\n\n    linha = {"config": nome}\n    for m in scoring:\n        vals = cv[f"test_{m}"]\n        print(f"  {m}: {vals.mean():.4f} ± {vals.std():.4f}")\n        linha[f"{m}_mean"] = vals.mean()\n        linha[f"{m}_std"] = vals.std()\n\n    resultados.append(linha)\n\n    # Matriz de confusão\n    y_pred_cv = cross_val_predict(pipe, X_tr, y_tr, cv=5)\n\n    cm = confusion_matrix(y_tr, y_pred_cv)\n\n    fig, ax = plt.subplots(figsize=(5, 5))\n    disp = ConfusionMatrixDisplay(confusion_matrix=cm)\n    disp.plot(cmap="Blues", ax=ax)\n    ax.set_title(f"Matriz de Confusão: 

In [7]:
import numpy as np
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay,
                             roc_auc_score)

pipe_final = Pipeline([
    ("prep", ColumnTransformer(configs['word+char'])),
    ("svm", LinearSVC(class_weight="balanced", random_state=42, max_iter=2000)),
])
pipe_final.fit(X_tr, y_tr)

y_pred = pipe_final.predict(X_te)
print(classification_report(y_te, y_pred, digits=4))
ConfusionMatrixDisplay.from_predictions(y_te, y_pred, cmap="Blues")

# LinearSVC não tem predict_proba; para AUC use decision_function
print("ROC-AUC:", roc_auc_score(y_te, pipe_final.decision_function(X_te)))

KeyboardInterrupt: 

In [13]:
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import GridSearchCV, StratifiedKFold

pipe_k = Pipeline([
    ("prep", ColumnTransformer(configs["word+char"])),
    ("select", SelectKBest(chi2, k=10_000)),
    ("svm", LinearSVC(class_weight="balanced", random_state=42, max_iter=2000)),
], memory="/content/cache_prep")   # evita refazer o TF-IDF a cada k

grid = GridSearchCV(
    pipe_k,
    param_grid={"select__k": [3000]},
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring="f1_macro", n_jobs=-1, return_train_score=True,
)
grid.fit(X_tr, y_tr)

res = pd.DataFrame(grid.cv_results_)[
    ["param_select__k", "mean_train_score", "mean_test_score", "std_test_score"]
]
print(res)

   param_select__k  mean_train_score  mean_test_score  std_test_score
0             3000          0.976908         0.955363        0.005514


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve, StratifiedKFold

cv_lc = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharey=True)

for ax, (nome, transformers) in zip(axes.ravel(), configs.items()):
    pipe = Pipeline([
        ("prep", ColumnTransformer(transformers)),
        ("svm", LinearSVC(class_weight="balanced", random_state=42, max_iter=2000)),
    ])

    train_sizes, train_scores, val_scores = learning_curve(
        pipe, X_tr, y_tr,
        train_sizes=np.linspace(0.1, 1.0, 6),
        cv=cv_lc,
        scoring="f1_macro",
        n_jobs=-1,
    )

    tr_m, tr_s = train_scores.mean(axis=1), train_scores.std(axis=1)
    va_m, va_s = val_scores.mean(axis=1), val_scores.std(axis=1)

    ax.plot(train_sizes, tr_m, "o-", label="Treino")
    ax.plot(train_sizes, va_m, "o-", label="Validação")
    ax.fill_between(train_sizes, tr_m - tr_s, tr_m + tr_s, alpha=0.15)
    ax.fill_between(train_sizes, va_m - va_s, va_m + va_s, alpha=0.15)
    ax.set_title(nome)
    ax.set_xlabel("Tamanho do treino")
    ax.set_ylabel("F1 macro")
    ax.legend()

plt.tight_layout()
plt.show()